# 5. Storage, recovery, and concurrency

The local backend is deliberately durable. SQLite stores the orchestration records; the filesystem store holds artifact bytes and task workspaces. A process may disappear, but the run should remain explainable and resumable.


## Local layout and store responsibilities

By default `.provium/pipeline.sqlite3` contains runs, tasks, dispatches, attempts, input sets, output mappings, artifact locations, and cache/retention-related records. `.provium/artifacts` contains managed objects; task workspaces are isolated from the store. `PROVIUM_PIPELINE_DATABASE` relocates the database and its sibling directories.

The library keeps protocol contracts separate from SQLite implementations so a future backend can preserve semantics without copying SQL details.


In [ ]:
from provium_pipeline.input.sqlite import SQLiteInputSetStore
from provium_pipeline.sqlite_attempts import SQLiteAttemptLeaseManager
from provium_pipeline.sqlite_dispatch_store import SQLiteDispatchStore
from provium_pipeline.sqlite_execution_store import SQLiteExecutionStore
from provium_pipeline.task_outputs import SQLiteTaskOutputStore

local_stores = (
    SQLiteExecutionStore, SQLiteDispatchStore, SQLiteAttemptLeaseManager,
    SQLiteInputSetStore, SQLiteTaskOutputStore,
)
assert all(local_stores)
local_stores


## Atomic boundaries

Compare-and-set transitions include the expected state. Two workers racing to claim or finish work cannot both silently win. Run creation stores the run and planned tasks transactionally. Output publication rejects a different value for an existing run/record/node/field key. File import uses stage → verify → atomic promote → index registration. These boundaries turn partial failures into detectable recovery cases.


## Recovery cases

- Crash before a lease: the ready task is still claimable.
- Crash during execution: the lease eventually expires and a new attempt can recover it.
- Crash after artifact promotion but before indexing: orphan scanning finds the bytes.
- Crash after publication but before final state transition: idempotent publication lets recovery observe the same output.
- Stale worker returns after lease takeover: its old token cannot commit.
- SQLite process restarts: durable run, task, dispatch, and attempt records reconstruct status.


In [ ]:
from provium_pipeline.multiprocessing import (
    MultiprocessSupervisor,
    WorkerProcess,
    WorkerRuntime,
)
from provium_pipeline.serial import SerialRunner

concurrency_boundaries = (
    SerialRunner, MultiprocessSupervisor, WorkerProcess, WorkerRuntime,
)
assert all(concurrency_boundaries)
concurrency_boundaries


## Serial versus multiprocess execution

The serial worker is the simplest reference execution path and is what the local CLI currently drives. Multiprocessing primitives isolate worker crashes and propagate cancellation, but they do not change store contracts or lease rules. The database remains the coordination authority; process memory is only a cache or convenience.

For backup, stop writers or use SQLite's backup mechanism and copy the artifact directory consistently. Never restore only one side when you need a complete operational snapshot.

**What to notice:** recovery is designed into identities, state transitions, leases, and publication—not bolted onto the worker. Next: [CLI workflows](06-cli-workflows.ipynb). Reference: [storage and recovery](../docs/storage-and-recovery.md).
